# Retrotraducción en línea (R6) — generar dentro de cada fold y evaluar

Genera los datos sintéticos **dentro del propio entrenamiento de cada fold** (sin armar archivos por fold a mano):
para cada fold se parafrasean (Helsinki-NLP) y se traducen al shiwilu (NLLB-200 + LoRA de F. Prado) las oraciones de su
pool, se filtran y se evalúa con la validación cruzada de 5 folds. Todo lo hace `validacion_cruzada_en_linea.py`.

**Estrategia (la misma que el entrenamiento): test / pool / core / dev.** Para cada fold: *test* = el fold (no se toca);
*pool* = los otros 4 folds; el pool se parte en *dev interno* (~1/6) y *core*. Hay dos etapas:
- **Etapa core (elegir `C`):** el aumento se genera/filtra solo con el core y se mide en el dev interno.
- **Etapa pool (modelo final):** el aumento se genera/filtra con todo el pool y se mide en el test.

Así, el dev interno nunca influye en el aumento con el que se elige `C`. En todas las etapas se descartan las filas sintéticas
idénticas a una oración del test.

En Retrotraducción cada fila sintética depende solo de su oración de origen, así que se genera **una vez por fold desde el pool**
y la etapa core usa las filas cuyo origen está en el core (equivale a generar solo con el core). Los filtros (idioma + semántico,
con el centroide LaBSE) se aplican en cada etapa con el centroide de **su** conjunto de entrenamiento.

**Antes de correr:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU** (T4 sirve).
2. En tu computadora: `git commit` + `git push` del repo **reorganizado** (carpetas `oe2_...`) con `validacion_cruzada_en_linea.py`.
   Esta Colab clona tu repo de GitHub.
3. Ten el checkpoint en Drive: `Mi unidad/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl/` (o entrénalo en el paso 3B).

**Tiempo estimado:** ~15-30 min de generación por fold (5 folds) + ~15 min de embeddings y evaluación.
**Si Colab se corta no pierdes lo hecho:** lo generado se guarda en Drive (`shiwilu_en_linea/`) a medida que se produce y, al volver a
correr el paso 4, se reutiliza.

**Nota sobre "bajar el repo de F. Prado":** se clona una sola vez (paso 1); la corrida lo usa en cada fold.

## 0. Montar Google Drive (aquí queda guardado lo generado)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repo y el de F. Prado

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git clone https://github.com/fapi19/Tesis_Spa-Jeb.git oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
!git log --oneline -3

ruta = '/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py'
assert os.path.exists(ruta), ('No existe validacion_cruzada_en_linea.py: haz git commit + git push del repo reorganizado '
                              '(carpetas oe2_...) con los cambios nuevos y vuelve a correr esta celda.')
print('OK')

## 2. Entorno aislado de F. Prado (sus versiones fijadas; evita conflictos con los paquetes de Colab)

In [ ]:
%cd /content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
!pip install -q virtualenv
!rm -rf /content/nmt_venv
!virtualenv -q /content/nmt_venv
!/content/nmt_venv/bin/pip install -q -r requirements/nmt.txt
!/content/nmt_venv/bin/pip install -q sentence-transformers scikit-learn matplotlib pandas
!/content/nmt_venv/bin/python -c "import torch; print('GPU disponible:', torch.cuda.is_available())"
%cd /content/shiwilu-tesis

## 3A. Copiar el checkpoint desde Drive al disco local (recomendado)

In [ ]:
CHECKPOINT = "/content/checkpoint_retro/nllb_bidi_lora_v2_1b_loraplus_xl"

!mkdir -p /content/checkpoint_retro
!cp -r "/content/drive/MyDrive/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl" /content/checkpoint_retro/
for f in ["adapter_model.safetensors", "adapter_config.json", "tokenizer_config.json"]:
    assert os.path.exists(os.path.join(CHECKPOINT, f)), f"Falta {f} en {CHECKPOINT}."
tam = os.path.getsize(os.path.join(CHECKPOINT, "adapter_model.safetensors")) / 1e9
assert tam > 1.5, f"adapter_model.safetensors pesa solo {tam:.2f} GB (deberia ser ~1.97 GB): copia incompleta."
print(f"OK: checkpoint completo ({tam:.2f} GB) en {CHECKPOINT}")

## 3B. (OPCIONAL) Entrenar el checkpoint desde cero — SOLO si NO lo tienes en Drive

Salta al paso 4 si hiciste el 3A. Entrenar la configuración campeona (v2.1b LoRA+) tarda 30 min - 2 h.
**Está protegido:** con `ENTRENAR = False` (por defecto) esta celda no hace nada, así que "Ejecutar todo" es seguro.

In [ ]:
ENTRENAR = False   # pon True SOLO si no tienes el checkpoint en Drive (y entonces omite el paso 3A)

if ENTRENAR:
    %cd /content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
    !/content/nmt_venv/bin/python -m scripts.nmt.30_train_lora --variant xl --rank 32 --alpha 64 --loraplus-lr-ratio 16 --output-dir models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl
    !mkdir -p /content/drive/MyDrive/shiwilu_checkpoint
    !cp -r models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl /content/drive/MyDrive/shiwilu_checkpoint/
    CHECKPOINT = "/content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb/models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl"
    %cd /content/shiwilu-tesis
else:
    print("3B omitido (ENTRENAR = False).")

## 4. Generar y evaluar, fold por fold (todo en una corrida)

Para cada fold: calcula test / pool / core / dev, parafrasea y traduce el pool (guarda `fold<N>_pool.csv` en Drive), filtra por etapa,
elimina copias del test, extrae embeddings de los 3 modelos y entrena (elige `C` con el core+dev, reentrena con el pool, predice el test).
El paso de traducción **no imprime progreso**: puede pasar mucho rato sin mostrar nada, y eso no significa que esté colgado.

In [ ]:
import os, subprocess

def correr(comando, cwd="/content/shiwilu-tesis"):
    """Corre un comando mostrando la salida en vivo; se detiene si falla."""
    proc = subprocess.Popen(
        comando, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"},
    )
    for linea in proc.stdout:
        print(linea, end="")
    proc.wait()
    assert proc.returncode == 0, f"Fallo (codigo {proc.returncode}): {' '.join(comando)}"

assert "CHECKPOINT" in globals(), "Falta correr el paso 3A (o 3B): define la variable CHECKPOINT."
PYTHON_NMT = "/content/nmt_venv/bin/python"

PRUEBA = False   # True = prueba rapida: solo el fold 0, con cache y etiqueta aparte (no contamina la corrida completa)
CACHE = "/content/drive/MyDrive/shiwilu_en_linea" + ("_prueba" if PRUEBA else "")
ETQ = "colab" + ("_prueba" if PRUEBA else "")

correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py",
        "--tecnica", "retrotraduccion", "--checkpoint", CHECKPOINT, "--cache", CACHE, "--etiqueta", ETQ]
       + (["--folds", "0"] if PRUEBA else []))

## 5. Revisar lo generado (por fold, desde el pool)

In [ ]:
import glob
import pandas as pd
from IPython.display import display

for ruta in sorted(glob.glob(f"{CACHE}/retrotraduccion/fold*_pool.csv")):
    df = pd.read_csv(ruta)
    print(os.path.basename(ruta), len(df), "filas |", df.intencion.value_counts().sort_index().to_dict())
display(df.sample(8, random_state=0)[["espanol_original", "espanol", "shiwilu", "intencion"]])

## 6. Tablas y curvas ROC (las mismas que el repo)

El CSV de predicciones de esta corrida trae **los 12 experimentos** (la retrotraducción nueva + las otras tres configuraciones vigentes) con las
probabilidades por categoría, así que se pasa directo a `comparacion_pareada.py` y `curvas_roc.py`. Se generan: `comparacion_pareada_sin_puntuacion_<TAG>.csv`,
`curvas_roc_<TAG>.csv`, `curvas_roc_<TAG>.png` (grilla 3 × 4) y la carpeta `curvas_roc_<TAG>/` con una imagen por experimento.

In [ ]:
if not PRUEBA:
    import pandas as pd
    from IPython.display import display
    RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
    TAG = f"en_linea_retrotraduccion_{ETQ}"
    ENTRADA = f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv"

    # mismas tablas y figuras que el repo: comparaciones pareadas, curvas ROC (CSV + grilla + una imagen por experimento)
    correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/comparacion_pareada.py", "--entrada", ENTRADA, "--etiqueta", TAG])
    correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/curvas_roc.py", "--entrada", ENTRADA, "--etiqueta", TAG])

    from IPython.display import Image
    display(Image(f"{RES}/curvas_roc_{TAG}.png"))
    auc_macro = pd.read_csv(f"{RES}/curvas_roc_{TAG}.csv").query("categoria == 'macro'")
    display(auc_macro.pivot(index="modelo", columns="tecnica", values="auc").round(4))

if PRUEBA:
    print('Modo prueba: solo se evaluo el fold 0; las tablas y curvas ROC necesitan los 5 folds.')

## 7. Comparar con `sin_aumento` y con la retrotraducción vigente (catálogo único)

In [ ]:
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion")
from comparacion_pareada import diferencia_pareada

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
vigente = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion.csv")
P_vig = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion.csv")

def f1(tabla, tecnica):
    return tabla[tabla.tecnica == tecnica].set_index("modelo").f1_macro_agrupado
if not PRUEBA:
    nuevo = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion_en_linea_retrotraduccion_{ETQ}.csv")
    display(pd.DataFrame({
        "sin aumento (vigente)": f1(vigente, "sin_aumento"),
        "retro catalogo unico (vigente)": f1(vigente, "retrotraduccion"),
        "retro EN LINEA (esta corrida)": f1(nuevo, "retrotraduccion"),
    }).round(4))

    P_nuevo = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_en_linea_retrotraduccion_{ETQ}.csv")
    P = pd.concat([
        P_nuevo[P_nuevo.tecnica == "retrotraduccion"].assign(tecnica="retro_en_linea"),
        P_vig[P_vig.tecnica == "sin_aumento"],
        P_vig[P_vig.tecnica == "retrotraduccion"].assign(tecnica="retro_vigente"),
    ])
    rng = np.random.default_rng(42)
    filas = []
    for modelo in ["labse", "mbert", "xlmr"]:
        for a, b in [("retro_en_linea", "sin_aumento"), ("retro_en_linea", "retro_vigente")]:
            d, lo, hi = diferencia_pareada(P, (modelo, a), (modelo, b), rng)
            filas.append({"modelo": modelo, "comparacion": f"{a} - {b}", "diferencia_f1": round(d, 4),
                          "ic95_bajo": round(lo, 4), "ic95_alto": round(hi, 4), "distinguible_de_cero": not (lo <= 0 <= hi)})
    display(pd.DataFrame(filas))

if PRUEBA:
    print('Modo prueba: solo se evaluo el fold 0; la comparacion pareada necesita los 5 folds.')

## 8. Descargar el resultado

Un `.zip` con lo generado (`retrotraduccion/fold*_pool.csv`) y los resultados: predicciones y resumen (12 experimentos), comparaciones pareadas,
curvas ROC (CSV, grilla e imágenes por experimento). En tu computadora: lo generado va en `oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea/` y todo lo demás en
`oe2_aumento_de_datos/evaluacion/resultados/`. Lo generado también queda en Drive (`shiwilu_en_linea/`).

In [ ]:
TAG = f"en_linea_retrotraduccion_{ETQ}"

import glob, shutil
from google.colab import files

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
carpeta = "/content/salida_retro_en_linea"
shutil.rmtree(carpeta, ignore_errors=True)
os.makedirs(carpeta + "/resultados")
os.makedirs(carpeta + "/en_linea")
shutil.copytree(f"{CACHE}/retrotraduccion", carpeta + "/en_linea/retrotraduccion")
for ruta_ in glob.glob(f"{RES}/*{TAG}*"):      # CSV, PNG y la carpeta con una imagen por experimento
    destino = carpeta + "/resultados/" + os.path.basename(ruta_)
    shutil.copytree(ruta_, destino) if os.path.isdir(ruta_) else shutil.copy(ruta_, destino)
zip_path = shutil.make_archive("/content/salida_retro_en_linea", "zip", carpeta)
print(sorted(os.listdir(carpeta + "/resultados")))
files.download(zip_path)